# SafeShare — train and measure on a free Colab GPU

Run the cells in order. Nothing here touches your laptop's copy of the project: this
session clones the repo, trains, and saves the results to your Google Drive.

**The order matters.** Cell 4 measures the model **as it is, before any training** — that
is the "before" column. Training starts at cell 5. If the baseline is already good, that is
a result to report, not a problem.

| step | cell | time on a T4 | can be skipped? |
|---|---|---|---|
| check the GPU, install, clone | 0–2 | 5 min | no |
| download the base model (once per session) | 3 | 5 min | no |
| build the training data | 3 | 1 min | no |
| **baseline measurement (the "before")** | 4 | 5–10 min | no |
| fine-tune (LoRA / QLoRA) | 5 | ~20 min | yes |
| measure after fine-tuning | 6 | 5–10 min | yes |
| align (DPO) | 7 | ~25 min | yes |
| measure after alignment | 8 | 5–10 min | yes |
| the comparison table | 9 | seconds | no |
| save everything to Drive | 10 | seconds | **no — do this one** |

Set the GPU first: **Runtime → Change runtime type → T4 GPU**. Free Colab disconnects
after idle time, which is why cell 10 saves to Drive and the earlier cells save as they go.


In [ ]:
# 0 - is there really a GPU here?
!nvidia-smi

# You want 'Tesla T4' and about 15 GB. If it says no GPU: Runtime -> Change runtime type -> T4 GPU.


In [ ]:
# 1 - get the code
#
# The repository is private, so Colab needs permission to read it. Two ways:
#
#   (a) Recommended - add a GitHub token as a Colab secret named GITHUB_TOKEN:
#       sidebar key icon -> Add new secret -> Name: GITHUB_TOKEN
#       GitHub -> Settings -> Developer settings -> Personal access tokens -> Fine-grained
#       -> Repository access: only safeshare2 -> Permissions: Contents = Read-only
#   (b) Or make the repository public once. Verified: its history holds no secrets
#       (.env was never committed, and the only 'sk-' ever added was a placeholder).
#       Keeping it private is the safer choice for graded coursework.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    repo = f'https://{token}@github.com/HajarOga/safeshare2.git'
    print('using the GITHUB_TOKEN from Colab secrets')
except Exception:
    repo = 'https://github.com/HajarOga/safeshare2.git'
    print('no GITHUB_TOKEN secret found - this clone only works if the repository is public')

!git clone -q $repo /content/safeshare2 || echo 'CLONE FAILED - see the two options above'
%cd /content/safeshare2

# do not leave the token inside the cloned folder's git config
!git remote set-url origin https://github.com/HajarOga/safeshare2.git
!ls


In [ ]:
# 2 - install what the app and the training need
#
# Colab ships its own versions of torch, pandas and friends. This step may therefore print
# 'dependency resolver' warnings naming google-colab, cudf, dask or torchvision. Those are
# WARNINGS about Colab's own packages, not failures: what matters is the 'done' at the end.
# Two of them can be ignored outright (pandas, opentelemetry). The torchvision one only
# matters if a later cell dies with a torchvision error - the fix is in the last section.
!pip install -q -r requirements.txt -r training/requirements-train.txt
print('done')


In [ ]:
# 3 - the base model (once per session), and the training data
import os

# local model, and let it fetch the weights it needs
os.environ['SAFESHARE2_PROVIDER'] = 'local'
os.environ['SAFESHARE2_ALLOW_DOWNLOAD'] = '1'

# ~6 GB, once. The app will not start without it - there is no rules-only fallback.
!python -m scripts.setup_model --yes

# the data: our own documents, then the prompt/completion and chosen/rejected pairs
!python -m training.make_training_documents
!python -m training.prepare_data --skip-public
!python -m training.make_preference_pairs


In [ ]:
# 3a - mount Google Drive, so a disconnect cannot cost you the run
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/SafeShare/predictions
!test -d /content/drive/MyDrive/SafeShare && echo 'Drive is ready' || echo 'MOUNT FAILED - fix this before training'


In [ ]:
# 4 - THE BASELINE: the prompt alone, no training at all
# This is the "before" column. Save it to Drive as soon as it finishes.
!python -m evaluation.evaluate --live --out predictions/base.jsonl
!python -m evaluation.evaluate --score predictions/base.jsonl --json predictions/base.metrics.json
!cp predictions/base.jsonl predictions/base.metrics.json /content/drive/MyDrive/SafeShare/predictions/
print('baseline saved to Drive')

!test -d /content/drive/MyDrive/SafeShare || echo 'Drive is not mounted - re-run cell 3a, or these files are lost when Colab disconnects'


## Optional from here — skip the training and the table still works

If the baseline was already strong, you can stop after cell 4 and report that. The brief
still wants the adaptation step run, so continue when you have time.


In [ ]:
# 5 - fine-tune with LoRA / QLoRA  (~20 min)
!python -m training.train_lora
!mkdir -p /content/drive/MyDrive/SafeShare
!cp -r training/lora_adapter /content/drive/MyDrive/SafeShare/
print('adapter saved to Drive')


In [ ]:
# 6 - measure after fine-tuning
os.environ['SAFESHARE2_ADAPTER'] = 'training/lora_adapter'
!python -m evaluation.evaluate --live --out predictions/lora.jsonl
!python -m evaluation.evaluate --score predictions/lora.jsonl --json predictions/lora.metrics.json
!cp predictions/lora.jsonl predictions/lora.metrics.json /content/drive/MyDrive/SafeShare/predictions/
print('done')


In [ ]:
# 7 - align with DPO  (~25 min) - starts from the adapter above
os.environ.pop('SAFESHARE2_ADAPTER', None)   # training reads the adapter from --adapter, not the env
!python -m training.train_dpo
!cp -r training/dpo_adapter /content/drive/MyDrive/SafeShare/
print('aligned adapter saved to Drive')


In [ ]:
# 8 - measure after alignment
os.environ['SAFESHARE2_ADAPTER'] = 'training/dpo_adapter'
!python -m evaluation.evaluate --live --out predictions/dpo.jsonl
!python -m evaluation.evaluate --score predictions/dpo.jsonl --json predictions/dpo.metrics.json
!cp predictions/dpo.jsonl predictions/dpo.metrics.json /content/drive/MyDrive/SafeShare/predictions/
print('done')


In [ ]:
# 9 - the table for the report
!python -m evaluation.evaluate --compare predictions/base.jsonl predictions/lora.jsonl predictions/dpo.jsonl

# If you skipped the training, compare what you have:
# !python -m evaluation.evaluate --compare predictions/base.jsonl


In [ ]:
# 10 - save everything, then download if you want it locally
!mkdir -p /content/drive/MyDrive/SafeShare
!cp -r predictions /content/drive/MyDrive/SafeShare/ 2>/dev/null; true
!cp -r training/lora_adapter /content/drive/MyDrive/SafeShare/ 2>/dev/null; true
!cp -r training/dpo_adapter /content/drive/MyDrive/SafeShare/ 2>/dev/null; true
!ls -la /content/drive/MyDrive/SafeShare/

# To keep a copy on your laptop too: put training/lora_adapter into the project folder
# and run the app with SAFESHARE2_ADAPTER=training/lora_adapter (Windows: set SAFESHARE2_ADAPTER=...)


## If something goes wrong

- **"no GPU"** — Runtime → Change runtime type → T4 GPU, then re-run from cell 0.
- **Out of memory during training** — lower the length: `!python -m training.train_lora --max-len 1536`,
  and if it still fails, `--batch-size 1 --grad-accum 16`. Do not trim the policy block out of the
  prompt: that is the part carrying the rules.
- **Disconnected** — Google Drive already holds every predictions file and any adapter that finished.
  Re-run from cell 0; the tooling skips what exists. Training itself has no resume, so start that cell again.
- **The model will not start in cell 4** — the download in cell 3 must finish first; re-run cell 3.
- **Everything finished but the numbers look identical** — that is a legitimate result. Report it, and
  let the prompt-size argument (`docs/MODEL_ADAPTATION.md` §3) carry the case for the training step.
- **`dependency resolver` warnings after cell 2** — warnings about Colab's own packages, not your
  install. Ignore pandas and opentelemetry. If a later cell fails with a **torchvision** error, run
  `!pip install -q --upgrade torchvision` and re-run that cell.


## What to put in the report

The three columns from cell 9, quoted as they print — plus the prompt budget from
`docs/MODEL_ADAPTATION.md` §3: 1,617 characters of instructions + 1,632 of policy for every
798 characters of document. That is the cost the adaptation step is arguing against, and the
baseline column is what makes the argument honest.
